## Часть 3

In [15]:
import numpy as np
import pandas as pd
from scipy import stats

alpha = 0.05

### Корреляция

Источник данных: https://tochno.st/datasets/regions_collection - социально-экономические показатели регионов России

Задача: вычислить корреляцию между количеством безработных и количеством совершенных преступлений по регионам России

In [16]:
economics_df = pd.read_csv("data/data_regions_collection_102_v20250605.csv", sep=";")

Часть датасета с данными о количестве безработных (тыс. чел.) по регионам ('по данным выборочного обследования рабочей силы, в среднем за год')

In [17]:
unemployment_df = economics_df[(economics_df["section"] == 'Занятость и безработица')
             & (economics_df["subsection"] == 'В том числе: безработные')
             & (economics_df["object_level"] == "регион")][["indicator_name", "subsection", "object_name", "indicator_value", "year"]]
unemployment_df.sample(3)

,indicator_name,subsection,object_name,indicator_value,year
1179902,Численность рабочей силы,В том числе: безработные,Псковская область,14.941944,2021
1180342,Численность рабочей силы,В том числе: безработные,Республика Саха (Якутия),34.442186,2021
1180286,Численность рабочей силы,В том числе: безработные,Иркутская область,68.841753,2021


Группировка числа безработных по регионам (в среднем за представленные года):

In [18]:
unemployment_grouped = unemployment_df.groupby("object_name")["indicator_value"].mean()
unemployment_grouped[:3]

object_name
Алтайский край           51.886770
Амурская область         18.759871
Архангельская область    33.729894
Name: indicator_value, dtype: float64

Группировка количества лиц, потерпевших преступления, по регионам (в среднем за представленные года):

In [19]:
crime = economics_df[(economics_df["subsection"] == "Число лиц, потерпевших от преступных посягательств")
                   & (economics_df["object_level"] == "регион")][["indicator_name", "object_name", "indicator_value", "year"]]
crime_grouped = crime.groupby("object_name")["indicator_value"].mean()
crime_grouped[:3]

object_name
Алтайский край           28165.5
Амурская область         12220.5
Архангельская область    12507.0
Name: indicator_value, dtype: float64

### Проверка корреляции

Проверим корреляцию при доверительных вероятностях $(\gamma)$ 95% и 99%

H0: Коэффициент корреляции незначимый.

H1: Коэффициент корреляции значимый.

In [20]:
def print_corr_results(results, test_type):
    print(f"========{test_type}========")
    print(f"Корреляция: {round(results.statistic, 3)}")
    print(f"p-value: {results.pvalue}")
    print(f"========{'=' * len(test_type)}========\n")

In [ ]:
# Пирсон
pearson_results = stats.pearsonr(unemployment_grouped, crime_grouped)
print_corr_results(pearson_results, "Пирсон")

# Спирмен
spearman_results = stats.spearmanr(unemployment_grouped, crime_grouped)
print_corr_results(spearman_results, "Спирмен")

# Кендалл
kendall_results = stats.kendalltau(unemployment_grouped, crime_grouped)
print_corr_results(kendall_results, "Кендалл")

# z-оценка для Кендалла
n = len(crime_grouped)
z_kendall = kendall_results.correlation * np.sqrt((9 * n) * (n + 1) / (2 * (2 * n + 5)))

print(f"z-статистика (Кендалл): {z_kendall}")
print(f"Табличное значение: {stats.norm.ppf(1 - 0.05 / 2)}\n")

# t-оценка для Спирмена
t_spearman = spearman_results.correlation * np.sqrt(n - 1)
print(f"t-статистика (Спирмен): {t_spearman}")
print(f"Табличное значение: {stats.t.ppf(1 - 0.05 / 2, df=n-2)}")

========Пирсон========
Корреляция: 0.709
p-value: 1.54885906637288e-14

========Спирмен========
Корреляция: 0.735
p-value: 5.123256616072145e-16

========Кендалл========
Корреляция: 0.627
p-value: 8.187207114247826e-18

z-статистика (Кендалл): 8.69632661700171
Табличное значение: 1.959963984540054

t-статистика (Спирмен): 6.818310194043743
Табличное значение: 1.9882679074772218


Проверка коэффициентов Кендалла и Спирмена:
Расчетные значения z выше табличных, поэтому H0 (коэффициент не значимый) отклоняется.

Все проведенные тесты показывают довольно высокую положительную корреляцию -- число безработных граждан коррелирует с количеством совершенных преступлений; низкое значение p-value подтверждает H0 при $\gamma = 0.95$ и $\gamma = 0.99$ -- коэффициенты корреляции значимы.

### Согласованность мнений экспертов

Источник данных: https://github.com/rfordatascience/tidytuesday/tree/main/data/2026/2026-03-10

In an online quiz, created as an independent project by Adam Kucharski, over 5,000 participants compared pairs of probability phrases (e.g. “Which conveys a higher probability: Likely or Probable?”) and assigned numerical values (0–100%) to each of 19 phrases. The resulting data can be used to analyse how people interpret common probability phrases.

Абсолютные оценки вероятностей, обозначаемых словами

In [22]:
abs_judgements_df = pd.read_csv("data/absolute_judgements.csv")
abs_judgements_df.sample(3)

,response_id,term,probability,order
59087,2068,Likely,75,17
6188,4852,Remote Chance,20,14
10167,4642,Will Happen,100,3


Попарные сравнения вероятностей, обозначаемых словами

In [23]:
pairwise_df = pd.read_csv("data/pairwise_comparisons.csv")
pairwise_df.sample(3)

,response_id,pair_id,term1,term2,selected
46970,480,1,Probable,Highly Unlikely,Probable
41482,1029,3,Remote Chance,Probable,Probable
30046,2173,7,Realistic Possibility,Almost Certain,Almost Certain


Данные о респондентах

In [24]:
resp_df = pd.read_csv("data/respondent_metadata.csv")
resp_df.sample(5)

,response_id,timestamp,age_band,english_background,education_level,country_of_residence
3356,1821,2026-01,25-34,English is my first language,High school,United Kingdom
1367,3810,2026-01,35-44,English is my first language,Bachelor,United States
706,4471,2026-02,35-44,English is my first language,Postgraduate,United States
595,4582,2026-02,45-54,English is my first language,Postgraduate,United States
412,4765,2026-02,25-34,English is my first language,Bachelor,United States


In [25]:
from scipy.stats import chi2
def kendall_concordance(df, feature_col="term", value_col="probability", expert_col="response_id"):
    """
    Вычисление коэффициента конкордации Кендалла.
    """

    m = df[expert_col].nunique() # кол-во экспертов
    n = df.groupby(expert_col)[value_col].count().values[0] # кол-во оцениваемых величин

    df["rank"] = df.groupby(expert_col)[value_col].rank(method="average", ascending=False).astype(int) # ранги для оценок каждого эксперта 
    rank_sum = df.groupby(feature_col)["rank"].sum() # сумма рангов для каждого слова
    rank_mean = rank_sum.sum() / n # средняя сумма рангов по всем словам
    delta_rank = rank_sum - rank_mean # отклонение суммы рангов каждого слова
    delta_rank_sq = np.square(delta_rank) # квадрат отклонения
    S = delta_rank_sq.sum()
    # Коэффициент W
    W = 12 * S / (m**2 * (n**3 - n))
    
    # Расчет статистики хи-квадрат
    chi2_stat = m * (n - 1) * W
    df_degrees = n - 1
    
    # Расчет p-value
    p_value = chi2.sf(chi2_stat, df_degrees) # выживательная функция (1 - cdf)
    
    return W, p_value


Вычисление ранговой согласованности мнений респондентов относительно значений вероятностей, определяемых представленными словами.

H0: коэффициент согласованности незначим

H1: коэффициент согласованности значим

In [26]:
W, p_value = kendall_concordance(abs_judgements_df)
print(f"""Коэффициент конкордации для оценок вероятностей, описываемых словами W = {round(W, 3)}, p-value = {p_value}""")

Коэффициент конкордации для оценок вероятностей, описываемых словами W = 0.875, p-value = 0.0


p-value < 0.05, поэтому H0 отвергается. Согласованность экспертов достаточно высокая (W > 0.8), что свидетельствует о схожем представлении о различии синонимичных слов среди людей с разным уровнем знания английского.

In [27]:
m = abs_judgements_df["response_id"].nunique() # кол-во экспертов
n = abs_judgements_df.groupby("response_id")["probability"].count().values[0] # кол-во оцениваемых величин
chi2 = m * (n - 1) * W # хи-квадрат
print(f"""Коэффициент хи-квадрат = {chi2}
          Кол-во экспертов = {m}
          Число степеней свободы = {n}
          Уровень значимости alpha = 0.05""")

Коэффициент хи-квадрат = 81459.65800149264
          Кол-во экспертов = 5174
          Число степеней свободы = 19
          Уровень значимости alpha = 0.05


In [30]:
alpha = 0.05
chi2_critical = stats.chi2.ppf(1 - alpha, n)
print(f"Хи-квадрат табличное: {chi2_critical}")

Хи-квадрат табличное: 30.14352720564616


Для n = 19 и $\alpha$ = 0.05 табличное значение $\chi^2$ = 30.1. H0 отвергается: расчетное значение сильно превышает табличное, поэтому коэффициент конкордации статистически значим.